# Liquidation research — TZ package

**Reviewer:** run this notebook **without** raw data — it displays bundled `results/`.

**Reproduce:** add parquet per `data/README.md`, then `python run_all.py` (features → strategies incl. **Bybit filter** → EDA).

See **[SUBMISSION.md](SUBMISSION.md)** · Spec: [docs/description.md](../docs/description.md)

Symbols: `btcusdt`, `ethusdt` · Dec 2025 – Feb 2026


## Review checklist (no data)

1. [EXECUTIVE_SUMMARY.md](../results/EXECUTIVE_SUMMARY.md)
2. **Bybit liquidation filter** → `results/strategies/bybit_filter_summary.csv` (below)
3. Full PnL → `results/strategies/PNL_REPORT.md`
4. EDA tiers 1.1–1.3 (sections below)
5. Optional chart: `ethusdt_price_liq_2025-12-15.png`


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display, Markdown, HTML

ROOT = Path('.').resolve()
if not (ROOT / 'config.py').exists():
    ROOT = Path('..').resolve()
import sys
sys.path.insert(0, str(ROOT))
from config import FIGURES, TABLES, DATA, SYMBOLS

REPO = ROOT.parent if (ROOT.parent / 'research').is_dir() else ROOT
STRAT = ROOT / 'results' / 'strategies'
PNL_TZ = STRAT / 'pnl_report.csv'
PNL_REPO = REPO / 'research' / 'pnl_evaluation' / 'results' / 'tables' / 'pnl_report.csv'
BUNDLE = REPO / 'research' / 'strategies_bundle'

print('ROOT:', ROOT)
print('DATA:', DATA, 'exists:', DATA.is_dir())
print('EDA tables:', len(list(TABLES.glob('*.csv'))), 'figures:', len(list(FIGURES.glob('*.png'))))
print('Strategy dir:', STRAT, 'exists:', STRAT.is_dir())


## Executive summary (EDA)


In [ ]:
p = ROOT / 'results' / 'EXECUTIVE_SUMMARY.md'
if p.exists():
    display(Markdown(p.read_text()))
else:
    print('Run: python run_all.py --eda-only')


## Tier summary


In [ ]:
t = TABLES / 'TIER_SUMMARY.csv'
if t.exists():
    display(pd.read_csv(t))
else:
    print('Missing TIER_SUMMARY.csv')


---
# Feature documentation


Микроструктурные фичи для 1s панели стратегий. Источник: `FEATURES.md` (копия bundle) + enriched parquet.

Сборка: `python build_features.py` → `data/enriched/`.


In [ ]:
feat_paths = [
    ROOT / 'FEATURES.md',
    BUNDLE / 'FEATURES.md',
]
feat_doc = next((p for p in feat_paths if p.is_file()), None)
if feat_doc:
    display(Markdown(feat_doc.read_text()))
else:
    print('FEATURES.md not found — run from tz_assignment/')


### Enriched parquet (проверка наличия)


In [ ]:
from config import enriched_paths
rows = []
for sym in SYMBOLS:
    ep = enriched_paths(sym)
    for k, path in ep.items():
        rows.append({'symbol': sym, 'stream': k, 'path': str(path), 'exists': path.is_file()})
display(pd.DataFrame(rows))


---
# Task 1.1 — Foundational EDA


### 1. Data quality & stream frequency


In [ ]:
display(pd.read_csv(TABLES / 'tier11_01_data_quality.csv').head(20))
display(pd.read_csv(TABLES / 'tier11_01b_stream_frequency.csv'))
display(Image(filename=str(FIGURES / 'tier11_01b_stream_frequency.png')))


### 2. Univariate & time-of-day


In [ ]:
display(pd.read_csv(TABLES / 'tier11_02_univariate_summary.csv'))
display(Image(filename=str(FIGURES / 'tier11_02_hour_of_day_liq.png')))
display(Image(filename=str(FIGURES / 'tier11_02_liq_notional_hist.png')))


### 3. Cross-exchange liquidations


In [ ]:
display(pd.read_csv(TABLES / 'tier11_03_cross_liq_alignment.csv'))
display(Image(filename=str(FIGURES / 'tier11_03_cross_liq_alignment.png')))


### 4. Price + liquidations (ETH sample day)


In [ ]:
p = FIGURES / 'ethusdt_price_liq_2025-12-15.png'
if p.exists():
    display(Image(filename=str(p)))
else:
    print('Missing — run: python plot_price_liq.py --symbol ethusdt --day 2025-12-15')


---
# Task 1.2 — Intermediate Analysis


### Broad market (spread, trades sample)


In [ ]:
display(pd.read_csv(TABLES / 'tier12_01_spread_quantiles.csv'))
display(pd.read_csv(TABLES / 'tier12_01_trade_stats_sample.csv'))
display(Image(filename=str(FIGURES / 'tier12_01_spread_distribution.png')))


### ML EDA: stationarity, regimes, outliers, ACF, signal-to-noise


In [ ]:
display(pd.read_csv(TABLES / 'tier12_02_stationarity_adf.csv'))
display(pd.read_csv(TABLES / 'tier12_02_vol_regimes.csv'))
display(pd.read_csv(TABLES / 'tier12_02_outlier_structure.csv'))
display(Image(filename=str(FIGURES / 'tier12_02_vol_regimes.png')))
display(Image(filename=str(FIGURES / 'tier12_02_autocorrelation.png')))
sn = TABLES / 'tier12_02_signal_to_noise.csv'
if sn.exists():
    display(pd.read_csv(sn))


---
# Task 1.3 — Advanced Exploration


### Orderbook at liquidation & side imbalance


In [ ]:
display(pd.read_csv(TABLES / 'tier13_01_depth_at_liq_sample.csv').describe())
display(pd.read_csv(TABLES / 'tier13_02_side_imbalance_liq.csv'))
display(Image(filename=str(FIGURES / 'tier13_02_side_imbalance.png')))


### Liquidation heatmaps (hour × day-of-week)


In [ ]:
for sym in SYMBOLS:
    p = FIGURES / f'tier13_03_liq_heatmap_{sym}.png'
    if p.exists():
        print(sym)
        display(Image(filename=str(p)))


### Cross-exchange lead-lag & div proxy


In [ ]:
display(pd.read_csv(TABLES / 'tier13_04_leadlag_all.csv'))
display(pd.read_csv(TABLES / 'tier13_04_cross_venue_div_proxy.csv'))
for sym in SYMBOLS:
    p = FIGURES / f'tier13_04_leadlag_heatmap_{sym}.png'
    if p.exists():
        display(Image(filename=str(p)))


---
# Strategies & PnL


## Bybit liquidation filter (bundled)

**Not** the official submission metric (that is **Binance trades** in [description.md](../docs/description.md)).

Stress test: **Bybit liquidations** as events, **Binance** signal, filter adverse flow, markout on **Binance mid**.

Use **`score_bps`** = improvement vs keeping all liq events.


In [ ]:
bybit_csv = STRAT / 'bybit_filter_summary.csv'
if bybit_csv.is_file():
    display(pd.read_csv(bybit_csv))
else:
    print('Run: python run_strategies.py')


### Strategy docs & Binance direction WR


Краткое описание: `STRATEGIES.md`. Прогон: `python run_strategies.py` / `python run_all.py`.


In [ ]:
strat_doc = ROOT / 'STRATEGIES.md'
if strat_doc.is_file():
    display(Markdown(strat_doc.read_text()))
else:
    print('Missing STRATEGIES.md')


### Run metadata (`results/strategies/run_meta.json`)


In [ ]:
meta_p = STRAT / 'run_meta.json'
if meta_p.is_file():
    meta = json.loads(meta_p.read_text())
    display(pd.DataFrame([meta]).T.rename(columns={0: 'value'}))
    print('Source:', meta_p)
else:
    print('No TZ strategy run yet. Use: python run_strategies.py')


### Direction accuracy (WR / hit_rate on test)


In [ ]:
ref_p = BUNDLE / 'artifacts' / 'metrics_reference.json'
if ref_p.is_file():
    print('Reference WR (full research):')
    display(pd.DataFrame(json.loads(ref_p.read_text())).T.head(10))

dir_csvs = sorted(STRAT.glob('direction_metrics_*.csv')) if STRAT.is_dir() else []
if not dir_csvs:
    print('No direction_metrics_*.csv in', STRAT)
else:
    for p in dir_csvs:
        sym = p.stem.replace('direction_metrics_', '')
        df = pd.read_csv(p)
        mtime = pd.Timestamp(p.stat().st_mtime, unit='s')
        print(f'\n=== {sym} (from {p.name}, mtime={mtime}) ===')
        pivot = df.pivot_table(index='horizon_sec', columns='model', values='hit_rate', aggfunc='first')
        display(pivot.style.format('{:.3f}').background_gradient(cmap='RdYlGn', vmin=0.48, vmax=0.58))
        display(df)


### Full maker PnL (Binance trades + Bybit liq, τ = 30 / 120 / 300 s)


In [ ]:
pnl_csv = PNL_TZ if PNL_TZ.is_file() else (PNL_REPO if PNL_REPO.is_file() else None)
pnl_md_paths = [STRAT / 'PNL_REPORT.md', REPO / 'research' / 'pnl_evaluation' / 'results' / 'PNL_REPORT.md']

if pnl_csv:
    src_label = 'TZ results/strategies' if pnl_csv == PNL_TZ else 'research/pnl_evaluation (latest full run)'
    print('PnL table:', pnl_csv, '|', src_label)
    pnl = pd.read_csv(pnl_csv)
    display(Markdown(f'**Rows:** {len(pnl)} · **Symbols:** ' + str(pnl['symbol'].unique().tolist())))

    # Top score_bps per symbol × venue (τ=30s primary)
    sub = pnl[pnl['tau_sec'] == 30].copy()
    sub = sub[sub['strategy'] != 'baseline'].sort_values('score_bps', ascending=False)
    print('\nTop score_bps @ τ=30s (excl. baseline):')
    display(sub.groupby(['symbol', 'venue'], as_index=False).head(3))

    for sym in sorted(pnl['symbol'].unique()):
        for venue in sorted(pnl['venue'].unique()):
            s = pnl[(pnl.symbol == sym) & (pnl.venue == venue) & (pnl.tau_sec == 30)]
            if s.empty:
                continue
            print(f'\n{sym} / {venue} @ 30s')
            cols = ['strategy', 'score_bps', 'pnl_kept_bps', 'winrate_kept', 'turnover_kept_usd_day', 'meets_turnover_constraint']
            display(s[cols].sort_values('score_bps', ascending=False))
else:
    print('No pnl_report.csv — run: python run_strategies.py (or full run_report.py in main repo)')


In [ ]:
for p in pnl_md_paths:
    if p.is_file():
        print('Markdown report:', p)
        display(Markdown(p.read_text()))
        break
else:
    print('No PNL_REPORT.md found')


---
## Reproducibility

```bash
cd tz_assignment
pip install -r requirements.txt -r requirements-strategies.txt
# optional: export LIQUIDATION_DATA_ROOT=/path/to/data
python run_all.py --view-only   # no data: list bundled paths
python run_all.py               # full reproduce when data present
python build_notebook.py
jupyter notebook notebooks/Liquidation_EDA_TZ.ipynb
```
